In [1]:
# 1. Imports and configuration
import os
import re
import time
import smtplib
from email.message import EmailMessage

import gspread
from dotenv import load_dotenv
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

load_dotenv()

GMAIL_ADDRESS = os.getenv("GMAIL_ADDRESS")
GMAIL_APP_PASSWORD = os.getenv("GMAIL_APP_PASSWORD")

if not GMAIL_ADDRESS or not GMAIL_APP_PASSWORD:
    raise ValueError("Add GMAIL_ADDRESS and GMAIL_APP_PASSWORD to your .env file.")

MODEL_NAME = "qwen3:1.7b"
COMPANY_NAME = "Gradiant Technologies"
CHECK_INTERVAL_SECONDS = 30


In [ ]:
# 2. Connect to Google Sheets
SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]
creds = None

if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file("token.json", SCOPES)

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            "credentials.json", SCOPES
        )
        creds = flow.run_local_server(port=0)

    with open("token.json", "w", encoding="utf-8") as token_file:
        token_file.write(creds.to_json())

gc = gspread.authorize(creds)

# Replace with your existing Google Sheets URL
SPREADSHEET_URL = "YOUR_GOOGLE_SHEET_URL"
spreadsheet = gc.open_by_url(SPREADSHEET_URL)
worksheet = spreadsheet.worksheet("Form responses 1")

# Add a status column so already processed candidates are skipped
headers = worksheet.row_values(1)
if "Email Status" not in headers:
    worksheet.update_cell(1, len(headers) + 1, "Email Status")
    headers = worksheet.row_values(1)

STATUS_COL = headers.index("Email Status") + 1


In [3]:
# 3. Define the email tools

@tool
def validate_email_details(recipient: str, subject: str, body: str) -> str:
    """Validate the recipient email, subject, and message body."""

    recipient = recipient.strip()

    if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", recipient):
        return f"Invalid email address: {recipient!r}"

    if not subject.strip():
        return "Email subject is missing."

    if not body.strip():
        return "Email body is missing."

    return "Email details are valid."



@tool
def draft_email(
    recipient: str,
    candidate_name: str,
    company_name: str,
    role: str,
    location: str
) -> str:
    """Draft an application-received confirmation email for a candidate."""
    subject = f"Application Received | {role}"
    body = f"""Dear {candidate_name},

Thank you for applying for the {role} position at {company_name}.

We have received your application and will review your details.
Our recruitment team will contact you regarding the next steps.

Regards,
Talent Acquisition Team
{company_name}""".strip()

    return f"To: {recipient}\nSubject: {subject}\n\n{body}"


@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """Send a validated email through Gmail SMTP."""
    validation = validate_email_details.invoke({
        "recipient": recipient,
        "subject": subject,
        "body": body
    })
    if validation != "Email details are valid.":
        return validation

    message = EmailMessage()
    message["From"] = GMAIL_ADDRESS
    message["To"] = recipient
    message["Subject"] = subject
    message.set_content(body)

    try:
        with smtplib.SMTP("smtp.gmail.com", 587, timeout=30) as server:
            server.starttls()
            server.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
            server.send_message(message)
        return "Email accepted by Gmail SMTP."
    except Exception as error:
        return f"Email sending failed: {type(error).__name__}: {error}"


In [4]:
# 4. Create the LangChain email agent
llm = ChatOllama(model=MODEL_NAME, temperature=0)

agent = create_agent(
    model=llm,
    tools=[validate_email_details, draft_email, send_email],
    system_prompt=f"""
You are an email automation assistant for {COMPANY_NAME}.
- Draft professional application-received emails using draft_email.
- Validate email details before sending.
- Use send_email only when explicitly instructed to send.
- Never invent candidate details or claim an email was sent unless
  the send_email tool reports success.
"""
)


In [ ]:
# 5. Continuously process new Google Form responses

def is_duplicate_email(email, current_row):
    email = email.strip().lower()
    records = worksheet.get_all_records()

    # Check only rows before the current row
    for existing_row_num, candidate in enumerate(records, start=2):
        if existing_row_num >= current_row:
            break

        existing_email = str(
            candidate.get("Email Address", "")
        ).strip().lower()

        if existing_email == email:
            return True

    return False


def process_new_candidates():
    records = worksheet.get_all_records()

    for row_num, candidate in enumerate(records, start=2):

        # Read current processing status
        current_status = str(
            worksheet.cell(row_num, STATUS_COL).value or ""
        ).strip()

        # Skip rows that have already been processed
        if current_status in ("Sent", "Duplicate Email"):
            continue

        name = str(candidate.get("Full Name", "")).strip()
        recipient = str(
            candidate.get("Email Address", "")
        ).strip()
        role = str(candidate.get("Job Role", "")).strip()
        location = str(
            candidate.get("Current Location", "Not specified")
        ).strip()

        # Validate required details
        if not name or not role:
            if current_status != "Missing Details":
                worksheet.update_cell(
                    row_num, STATUS_COL, "Missing Details"
                )
                print(f"Missing details in row {row_num}")
            continue

        # Validate email format
        if not re.fullmatch(
            r"[^@\s]+@[^@\s]+\.[^@\s]+",
            recipient
        ):
            if current_status != "Invalid Email":
                worksheet.update_cell(
                    row_num, STATUS_COL, "Invalid Email"
                )
                print(f"Invalid email at row {row_num}: {recipient}")
            continue

        # Check for duplicate email before sending
        if is_duplicate_email(recipient, row_num):
            worksheet.update_cell(
                row_num, STATUS_COL, "Duplicate Email"
            )
            print(f"Duplicate email skipped: {recipient}")
            continue

        subject = f"Application Received | {role}"

        body = f"""Dear {name},

Thank you for applying for the {role} position at {COMPANY_NAME}.

We have received your application and will review your details.
Our recruitment team will contact you regarding the next steps.

Regards,
Talent Acquisition Team
{COMPANY_NAME}""".strip()

        try:
            result = send_email.invoke({
                "recipient": recipient,
                "subject": subject,
                "body": body
            })

            if result == "Email accepted by Gmail SMTP.":
                worksheet.update_cell(
                    row_num, STATUS_COL, "Sent"
                )
                print(f"Sent confirmation to {recipient}")

            else:
                worksheet.update_cell(
                    row_num, STATUS_COL, "Failed"
                )
                print(f"{recipient}: {result}")

        except Exception as error:
            worksheet.update_cell(
                row_num, STATUS_COL, "Failed"
            )
            print(f"Failed for {recipient}: {error}")


# Keep running to monitor submissions every 30 seconds.
# Stop using the notebook's interrupt button.
while True:
    process_new_candidates()
    time.sleep(CHECK_INTERVAL_SECONDS)